# Improved Baseline: TF-IDF + SVM with Optimisations

This notebook builds on `02_baseline_models.ipynb` and applies **all four improvements** described in `02_improvements.md`:

| # | Improvement | Key Idea |
|---|-------------|----------|
| 1 | **Hyperparameter Optimisation** | GridSearchCV over `C`, `loss`, `min_df` |
| 2 | **Advanced Feature Engineering** | FeatureUnion combining char n-grams + word n-grams |
| 3 | **Stratified Cross-Validation** | 5-fold StratifiedKFold for robust evaluation |
| 4 | **Stopword Removal** | Spanish stopwords removed before TF-IDF |

**Baseline reference:** Accuracy = 0.5693, Macro-F1 = 0.5091

## 1. Setup

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make the src module importable from the notebooks folder
sys.path.insert(0, os.path.abspath('../src'))

from data_processing import (
    normalize_text, normalize_texts,
    extract_category,
    prepare_category_dataset, split_category_dataset,
)
from evaluation import (
    calculate_category_metrics, print_classification_report,
    print_comparison_table, plot_comparison,
    generate_submission,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate,
)

# Improvement 4: Spanish stopwords
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
SPANISH_STOPWORDS = stopwords.words('spanish')

RANDOM_STATE = 42
print(f'Setup complete. Spanish stopwords loaded: {len(SPANISH_STOPWORDS)} words.')

## 2. Data Loading & Preprocessing

In [ ]:
codif_df = pd.read_csv('../data/codification_data.csv')
lead_df  = pd.read_csv('../data/leaderboard_data.csv')

print(f'Training rows  : {len(codif_df):,}')
print(f'Leaderboard    : {len(lead_df):,}')

In [ ]:
# Prepare single-label category dataset (majority-vote per literal)
cat_df = prepare_category_dataset(
    codif_df,
    literal_col='Literal',
    code_col='Code',
)

# Normalise text (lowercase, strip accents, remove punctuation/digits)
cat_df['Literal'] = normalize_texts(cat_df['Literal'])
print(f'\nDataset ready: {len(cat_df):,} unique literals, {cat_df["y_category"].nunique()} categories')

In [ ]:
# Train / Validation split (same as baseline for fair comparison)
X_train, X_val, y_train, y_val = split_category_dataset(
    cat_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

---
## 3. Improvement 4 — Stopword Removal

Before building TF-IDF features, we remove common Spanish stopwords that add noise to the feature matrix.

> *"Removing stopwords reduces the dimensionality and noise, allowing the SVM to focus on clinical terms."*

In [ ]:
def remove_stopwords(texts, sw_list):
    """Remove stopwords from a list of already-normalised texts."""
    result = []
    sw_set = set(sw_list)
    for text in texts:
        tokens = text.split()
        filtered = [t for t in tokens if t not in sw_set]
        result.append(' '.join(filtered))
    return result

# Normalise stopwords the same way as our text pipeline
norm_stopwords = [normalize_text(sw) for sw in SPANISH_STOPWORDS]
norm_stopwords = [sw for sw in norm_stopwords if sw]  # remove empty

X_train_clean = remove_stopwords(X_train, norm_stopwords)
X_val_clean   = remove_stopwords(X_val, norm_stopwords)

# Show effect
print('Stopword removal examples:')
for orig, clean in list(zip(X_train[:5], X_train_clean[:5])):
    if orig != clean:
        print(f'  "{orig}"  →  "{clean}"')

---
## 4. Improvement 2 — FeatureUnion (Char + Word TF-IDF)

Instead of using only character n-grams, we **combine** two TF-IDF vectorisers:
- **`char_tfidf`**: Character n-grams `(3, 6)` with `analyzer='char_wb'` — captures morphological/structural patterns.
- **`word_tfidf`**: Word n-grams `(1, 2)` with `analyzer='word'` — captures exact medical terms and bigrams.

Both include Spanish stopword filtering.

In [ ]:
# Build the FeatureUnion + SVM Pipeline
feature_union = FeatureUnion([
    ('char', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 6),
        sublinear_tf=True,
        max_features=100_000,
        min_df=2,
        dtype=np.float32,
    )),
    ('word', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_features=50_000,
        min_df=2,
        dtype=np.float32,
    )),
])

improved_pipeline = Pipeline([
    ('features', feature_union),
    ('clf', LinearSVC(
        C=1.0,
        max_iter=10_000,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    )),
])

print('Pipeline structure:')
print(improved_pipeline)

### 4.1 Quick Validation — FeatureUnion vs Baseline (before tuning)

In [ ]:
print('Training FeatureUnion + SVM (default C=1.0)...')
t0 = time.time()
improved_pipeline.fit(X_train_clean, y_train)
t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')

y_pred_val = improved_pipeline.predict(X_val_clean)
metrics_fu = calculate_category_metrics(y_val, y_pred_val)

print('\n--- FeatureUnion + Stopwords (pre-tuning) ---')
for k, v in metrics_fu.items():
    print(f'  {k:<25}: {v:.4f}')

print(f'\n  Baseline accuracy : 0.5693  →  Now: {metrics_fu["accuracy"]:.4f}  ({"+" if metrics_fu["accuracy"] > 0.5693 else ""}{(metrics_fu["accuracy"] - 0.5693)*100:.2f}pp)')
print(f'  Baseline macro_f1 : 0.5091  →  Now: {metrics_fu["macro_f1"]:.4f}  ({"+" if metrics_fu["macro_f1"] > 0.5091 else ""}{(metrics_fu["macro_f1"] - 0.5091)*100:.2f}pp)')

---
## 5. Improvement 1 — Hyperparameter Optimisation (GridSearchCV)

We now systematically search for the best combination of:
- **SVM `C`**: `[0.1, 1.0, 10.0]`
- **SVM `loss`**: `['hinge', 'squared_hinge']`
- **Char TF-IDF `min_df`**: `[1, 2, 5]`

The search uses **3-fold Stratified CV** on the training set, optimising for `f1_macro`.

In [ ]:
param_grid = {
    'clf__C': [0.1, 1.0, 10.0],
    'clf__loss': ['hinge', 'squared_hinge'],
    'features__char__min_df': [1, 2, 5],
}

grid_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    improved_pipeline,
    param_grid,
    cv=grid_cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print('Running GridSearchCV (3×6 = 18 fits per fold, 54 total)...')
t0 = time.time()
grid_search.fit(X_train_clean, y_train)
t1 = time.time()
print(f'\nGrid search completed in {t1 - t0:.1f}s')

In [ ]:
print('=== GridSearchCV Results ===')
print(f'Best parameters : {grid_search.best_params_}')
print(f'Best CV macro-F1: {grid_search.best_score_:.4f}')

# Show top-5 configurations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')
top5 = results_df[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(5)
print('\nTop-5 configurations:')
display(top5)

In [ ]:
# Evaluate best model on the held-out validation set
best_pipeline = grid_search.best_estimator_
y_pred_val_tuned = best_pipeline.predict(X_val_clean)
metrics_tuned = calculate_category_metrics(y_val, y_pred_val_tuned)

print('--- Best Tuned Model on Validation Set ---')
for k, v in metrics_tuned.items():
    print(f'  {k:<25}: {v:.4f}')

print(f'\n  Baseline accuracy : 0.5693  →  Tuned: {metrics_tuned["accuracy"]:.4f}  ({"+" if metrics_tuned["accuracy"] > 0.5693 else ""}{(metrics_tuned["accuracy"] - 0.5693)*100:.2f}pp)')
print(f'  Baseline macro_f1 : 0.5091  →  Tuned: {metrics_tuned["macro_f1"]:.4f}  ({"+" if metrics_tuned["macro_f1"] > 0.5091 else ""}{(metrics_tuned["macro_f1"] - 0.5091)*100:.2f}pp)')

In [ ]:
# Per-class classification report for the tuned model
print('Per-class classification report (tuned model):')
print_classification_report(y_val, y_pred_val_tuned)

---
## 6. Improvement 3 — Robust 5-Fold Stratified Cross-Validation

Instead of relying on a single 80/20 split, we evaluate the **best pipeline** with 5-fold Stratified CV on the **entire dataset** to get a more robust performance estimate.

> *"If the variance across folds is high, it indicates the model is sensitive to data distribution."*

In [ ]:
# Merge train + val for full cross-validation
X_all = cat_df['Literal'].tolist()
y_all = cat_df['y_category'].tolist()

# Apply stopword removal to entire dataset
X_all_clean = remove_stopwords(X_all, norm_stopwords)

# 5-Fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Running 5-fold Stratified Cross-Validation on full dataset...')
t0 = time.time()
cv_results = cross_validate(
    best_pipeline,
    X_all_clean, y_all,
    cv=skf,
    scoring=['accuracy', 'f1_macro'],
    return_train_score=False,
    n_jobs=-1,
)
t1 = time.time()
print(f'Done in {t1 - t0:.1f}s')

acc_scores = cv_results['test_accuracy']
f1_scores  = cv_results['test_f1_macro']

print(f'\n=== 5-Fold Cross-Validation Results ===')
print(f'Accuracy : {acc_scores.mean():.4f} ± {acc_scores.std():.4f}  (folds: {np.round(acc_scores, 4)})')
print(f'Macro-F1 : {f1_scores.mean():.4f} ± {f1_scores.std():.4f}  (folds: {np.round(f1_scores, 4)})')
print(f'\nBaseline (single-split): Accuracy=0.5693, Macro-F1=0.5091')
print(f'CV improvement        : Accuracy {"+" if acc_scores.mean() > 0.5693 else ""}{(acc_scores.mean() - 0.5693)*100:.2f}pp, Macro-F1 {"+" if f1_scores.mean() > 0.5091 else ""}{(f1_scores.mean() - 0.5091)*100:.2f}pp')

---
## 7. Final Comparison — Baseline vs Improved

In [ ]:
# Collect all results
comparison = {
    'Baseline (02_notebook)': {
        'accuracy': 0.5693,
        'weighted_f1': 0.5658,
        'macro_f1': 0.5091,
    },
    'FeatureUnion + Stopwords (pre-tune)': {
        'accuracy': metrics_fu['accuracy'],
        'weighted_f1': metrics_fu['weighted_f1'],
        'macro_f1': metrics_fu['macro_f1'],
    },
    'Best Tuned (GridSearch)': {
        'accuracy': metrics_tuned['accuracy'],
        'weighted_f1': metrics_tuned['weighted_f1'],
        'macro_f1': metrics_tuned['macro_f1'],
    },
    '5-Fold CV (mean ± std)': {
        'accuracy': acc_scores.mean(),
        'weighted_f1': np.nan,
        'macro_f1': f1_scores.mean(),
    },
}

comp_df = print_comparison_table(comparison)
plot_comparison({
    k: v for k, v in comparison.items() if k != '5-Fold CV (mean ± std)'
})

---
## 8. Leaderboard Submission with Best Model

Since the tuned model outperforms the baseline, we retrain on **100% of the data** and generate a new submission.

In [ ]:
# Retrain the best pipeline on the ENTIRE training dataset
print('Retraining best pipeline on full dataset...')
t0 = time.time()
best_pipeline.fit(X_all_clean, y_all)
t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')

# Normalise and clean leaderboard data
lead_literals_norm  = normalize_texts(lead_df['Literal'].tolist())
lead_literals_clean = remove_stopwords(lead_literals_norm, norm_stopwords)

# Predict
y_lead_pred = best_pipeline.predict(lead_literals_clean)
print(f'Predictions: {len(y_lead_pred):,} literals')
print(f'Unique categories predicted: {len(set(y_lead_pred))}')
print(f'Any empty: {sum(1 for p in y_lead_pred if not p)}')

In [ ]:
# Generate submission file
os.makedirs('../submissions', exist_ok=True)

submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path='../submissions/svm_improved.csv',
)

print('\nSubmission preview:')
display(submission_df.head(10))
print('\nCategory distribution in submission:')
print(submission_df['y_category'].value_counts().sort_index())

---
## 9. Summary of Improvements

| Improvement | What was done | Impact |
|-------------|---------------|--------|
| **1. GridSearchCV** | Tuned `C`, `loss`, `min_df` over 18 configs × 3 folds | Found optimal regularisation / loss |
| **2. FeatureUnion** | Combined char (3-6) + word (1-2) TF-IDF features | Captures both morphology and exact terms |
| **3. 5-Fold Stratified CV** | Evaluated on full data with 5 folds | Robust metric with confidence interval |
| **4. Stopword Removal** | Filtered Spanish stopwords from normalised text | Reduced noise in feature space |

**Output:** `submissions/svm_improved.csv` — columns: `id`, `Literal`, `y_category`